In [0]:
# XGBoost — the machine learning model we'll use for churn prediction
# SHAP — explains why the model made each prediction (tells us which factors matter most for churn)
%pip install xgboost shap

In [0]:
# Notebook 4: Churn Prediction
# Food Delivery Analysis

# Importing all the libraries
# --------------------------------------------

from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
import mlflow
import mlflow.xgboost
import mlflow.sklearn
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, accuracy_score
)
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load user features from Parquet
user_features = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/user_features.parquet"
)

print(f"Loaded {user_features.count():,} users with {len(user_features.columns)} features.")
user_features.display()

In [0]:
# Convert the PySpark dataframe to Pandas
# XGBoost and scikit-learn work with Pandas, not PySpark
user_pd = user_features.toPandas()

# Check the shape
print(f"Dataset shape: {user_pd.shape}")
print(f"\nChurn rate: {user_pd['churn_risk'].mean():.1%}")
print(f"\nColumn types:\n{user_pd.dtypes}")

In [0]:
# Drop user_id as it is just an identifier, not a feature
user_pd = user_pd.drop(columns=["user_id"])

# Convert text columns to numbers using label encoding
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col_name in ["city", "payment_method", "top_cuisine"]:
    user_pd[col_name] = le.fit_transform(user_pd[col_name])

# Separate features and target variable
X = user_pd.drop(columns=["churn_risk"])
y = user_pd["churn_risk"]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"\nFeature columns:\n{list(X.columns)}")

In [0]:
# Split data into training and test sets
# 80% of the data is used to train the model
# 20% is kept aside to test how well the model performs on unseen data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {X_train.shape[0]:,} users")
print(f"Test set size: {X_test.shape[0]:,} users")
print(f"\nChurn rate in training set: {y_train.mean():.1%}")
print(f"Churn rate in test set: {y_test.mean():.1%}")

In [0]:
# Train the XGBoost churn prediction model
# MLflow tracks everything automatically so we can review the results later

with mlflow.start_run(run_name="XGBoost Churn Prediction"):
    
    # Define the model and its settings
    xgb_model = XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss",
        use_label_encoder=False
    )
    
    # Train the model on the training set
    xgb_model.fit(X_train, y_train)
    
    # Make predictions on the test set
    y_pred = xgb_model.predict(X_test)
    y_pred_proba = xgb_model.predict_proba(X_test)[:, 1]
    
    # Calculate performance scores
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Log results to MLflow
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 6)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)
    mlflow.xgboost.log_model(xgb_model, "xgboost_churn_model")
    
    print(f"Accuracy: {accuracy:.1%}")
    print(f"ROC AUC Score: {roc_auc:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn Risk"]))

In [0]:
# Confusion matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Churn", "Churn Risk"],
            yticklabels=["No Churn", "Churn Risk"])
plt.title("XGBoost Churn Prediction - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/tmp/xgb_confusion_matrix.png", dpi=150)
plt.show()
print("Confusion matrix saved.")

In [0]:
# SHAP feature importance
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 10))
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.title("XGBoost Churn Prediction - Feature Importance (SHAP)")
plt.tight_layout()
plt.savefig("/tmp/xgb_shap_importance.png", dpi=150)
plt.show()
print("SHAP feature importance saved.")

In [0]:
# LSTM Data Preparation
# Reshape user order history into monthly sequences

# Installing PyTorch instead of TensorFlow for LSTM implementation.
# TensorFlow has a known compatibility conflict with Python 3.12 and thevpre-installed Protobuf version in Databricks Serverless. 
# PyTorch resolves this cleanly and is the more widely adopted deep learning framework for custom LSTM work in production environments
# ----------------------------------------------------------------------------------------------------------------------------------------

%pip install torch xgboost shap 

In [0]:
# Imports for LSTM using PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow
import mlflow.pytorch

print(f"PyTorch version: {torch.__version__}")

# Load the full dataset from Parquet
df_full = spark.read.parquet(
    "/Volumes/workspace/default/food_delivery_data/uae_food_delivery_750k.parquet"
)

# Convert to Pandas
df_pd = df_full.toPandas()

# Convert order_date to datetime and extract month
df_pd["order_date"] = pd.to_datetime(df_pd["order_date"])
df_pd["order_month"] = df_pd["order_date"].dt.month

print(f"Loaded {len(df_pd):,} rows")
print(f"Date range: {df_pd['order_date'].min()} to {df_pd['order_date'].max()}")

In [0]:
# Build monthly order sequences per user
# Each user gets 12 rows, one per month, with their activity for that month
# This is the format LSTM needs, a sequence of time steps per user

# monthly = df_pd.groupby(["user_id", "order_month"]).agg(
#     monthly_orders    = ("order_id", "count"),
#     monthly_spend     = ("total_price_aed", "sum"),
#     monthly_cancels   = ("order_status", lambda x: (x == "Cancelled").sum()),
#     avg_risk_score    = ("order_quality_risk_score", "mean"),
#     avg_delivery_time = ("delivery_duration_mins", "mean")
# ).reset_index()

# # Pivot to get one row per user with 12 monthly columns per feature
# # Then we reshape into a 3D array for LSTM input
# all_users = df_pd["user_id"].unique()
# churn_labels = df_pd.groupby("user_id")["churn_risk"].first().reset_index()

# print(f"Monthly sequences built for {monthly['user_id'].nunique():,} users")
# print(f"Months covered: {sorted(monthly['order_month'].unique())}")
# print(f"\nSample monthly data:")
# monthly.head(12).display()

# Build monthly order sequences per user using PySpark
# PySpark is much faster than Pandas for large aggregations in Databricks

from pyspark.sql.functions import col, count, sum, avg, month, when

# Add month column to the full Spark dataframe
df_monthly = df_full.withColumn("order_month", month(col("order_date"))) \
    .withColumn("is_cancelled", when(col("order_status") == "Cancelled", 1).otherwise(0))

# Aggregate at user and month level
monthly_spark = df_monthly.groupBy("user_id", "order_month").agg(
    count("order_id").alias("monthly_orders"),
    sum("total_price_aed").alias("monthly_spend"),
    sum("is_cancelled").alias("monthly_cancels"),
    avg("order_quality_risk_score").alias("avg_risk_score"),
    avg("delivery_duration_mins").alias("avg_delivery_time")
)

# Get churn label per user
churn_spark = df_monthly.groupBy("user_id").agg(
    avg("churn_risk").alias("churn_risk")
)

# Convert to Pandas only after aggregation
# Much smaller dataframe now, around 270,000 rows instead of 771,000
monthly = monthly_spark.toPandas()
churn_labels = churn_spark.toPandas()
churn_labels["churn_risk"] = churn_labels["churn_risk"].round().astype(int)

print(f"Monthly sequences built for {monthly['user_id'].nunique():,} users")
print(f"Months covered: {sorted(monthly['order_month'].unique())}")
print(f"Total monthly rows: {len(monthly):,}")

In [0]:
# Reshape monthly data into 3D sequences for LSTM
# Using vectorised numpy operations instead of loops for speed

N_MONTHS = 12
FEATURES = ["monthly_orders", "monthly_spend", "monthly_cancels",
            "avg_risk_score", "avg_delivery_time"]

# Get all unique users in consistent order
all_users = sorted(monthly["user_id"].unique())
n_users = len(all_users)
n_features = len(FEATURES)

# Create user to index mapping
user_index = {uid: idx for idx, uid in enumerate(all_users)}

# Add index columns for fast array filling
monthly["user_idx"]  = monthly["user_id"].map(user_index)
monthly["month_idx"] = monthly["order_month"].astype(int) - 1

# Initialise empty 3D array with zeros
X_seq = np.zeros((n_users, N_MONTHS, n_features))

# Fill using vectorised numpy indexing, no loops
X_seq[
    monthly["user_idx"].values,
    monthly["month_idx"].values
] = monthly[FEATURES].values

# Build target labels using dictionary lookup, much faster than iterating
churn_lookup = dict(zip(churn_labels["user_id"], churn_labels["churn_risk"]))
y_seq = np.array([churn_lookup[uid] for uid in all_users])

print(f"X_seq shape: {X_seq.shape}")
print(f"y_seq shape: {y_seq.shape}")
print(f"Churn rate in sequences: {y_seq.mean():.1%}")

In [0]:
# Scale the features to a 0 to 1 range
# LSTM learns much better when all features are on the same scale

# Reshape to 2D for scaling, then back to 3D
n_users, n_months, n_features = X_seq.shape
X_flat = X_seq.reshape(-1, n_features)

scaler = MinMaxScaler()
X_scaled_flat = scaler.fit_transform(X_flat)

# Reshape back to 3D
X_scaled = X_scaled_flat.reshape(n_users, n_months, n_features)

# Train test split, 80% train, 20% test
split = int(0.8 * n_users)
X_train = X_scaled[:split]
X_test  = X_scaled[split:]
y_train = y_seq[:split]
y_test  = y_seq[split:]

print(f"Training set: {X_train.shape[0]:,} users")
print(f"Test set: {X_test.shape[0]:,} users")
print(f"Churn rate in training set: {y_train.mean():.1%}")
print(f"Churn rate in test set: {y_test.mean():.1%}")

In [0]:
# Build the LSTM model using PyTorch

class ChurnLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(ChurnLSTM, self).__init__()
        
        # LSTM layer, reads through the 12 monthly time steps
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        
        # Fully connected output layer
        # Takes the final LSTM output and produces a single churn probability
        self.fc = nn.Linear(hidden_size, 1)
        
        # Sigmoid activation converts output to a probability between 0 and 1
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # Pass sequence through LSTM
        lstm_out, _ = self.lstm(x)
        
        # Take only the output from the last time step (month 12)
        last_output = lstm_out[:, -1, :]
        
        # Pass through the output layer and sigmoid
        out = self.sigmoid(self.fc(last_output))
        return out

# Define model settings
INPUT_SIZE  = 5    # Number of features per time step
HIDDEN_SIZE = 64   # Number of LSTM memory units
NUM_LAYERS  = 2    # Number of stacked LSTM layers
DROPOUT     = 0.2  # Dropout rate to prevent overfitting

# Initialise the model
model = ChurnLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT)

print("LSTM Model Architecture:")
print(model)
total_params = 0
for p in model.parameters():
    total_params += p.numel()
print(f"\nTotal trainable parameters: {total_params:,}")

In [0]:
# Train the LSTM model using PyTorch

# Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train)
X_test_tensor  = torch.FloatTensor(X_test)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
y_test_tensor  = torch.FloatTensor(y_test).unsqueeze(1)

# Create DataLoader for batch training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader  = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Define loss function and optimiser
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# Training loop
N_EPOCHS = 20
train_losses = []

with mlflow.start_run(run_name="LSTM Churn Prediction"):

    mlflow.log_param("hidden_size", HIDDEN_SIZE)
    mlflow.log_param("num_layers", NUM_LAYERS)
    mlflow.log_param("dropout", DROPOUT)
    mlflow.log_param("epochs", N_EPOCHS)
    mlflow.log_param("batch_size", 64)
    mlflow.log_param("learning_rate", 0.001)

    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0

        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{N_EPOCHS}  |  Loss: {avg_loss:.4f}")

    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        y_pred_proba = model(X_test_tensor).numpy().flatten()
        y_pred = (y_pred_proba >= 0.5).astype(int)

    accuracy = accuracy_score(y_test, y_pred)
    roc_auc  = roc_auc_score(y_test, y_pred_proba)

    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("roc_auc", roc_auc)

    print(f"\nAccuracy:     {accuracy:.1%}")
    print(f"ROC AUC:      {roc_auc:.3f}")
    print(f"\nDetailed Report:")
    print(classification_report(y_test, y_pred, target_names=["No Churn", "Churn Risk"]))

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# LSTM Confusion Matrix
plt.figure(figsize=(8, 6))
cm_lstm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm_lstm, annot=True, fmt="d", cmap="Greens",
            xticklabels=["No Churn", "Churn Risk"],
            yticklabels=["No Churn", "Churn Risk"])
plt.title("LSTM Churn Prediction - Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("/tmp/lstm_confusion_matrix.png", dpi=150)
plt.show()
print("LSTM confusion matrix saved.")

In [0]:
# Plot the training loss curve over 20 epochs
plt.figure(figsize=(8, 5))
plt.plot(range(1, N_EPOCHS + 1), train_losses, color="steelblue", linewidth=2, marker="o", markersize=4)
plt.title("LSTM Training Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/tmp/lstm_training_loss.png", dpi=150)
plt.show()
print("Training loss curve saved.")

In [0]:
# Model comparison summary
print("-" * 55)
print("CHURN PREDICTION MODEL COMPARISON SUMMARY")
print("-" * 55)
print(f"{'Model':<20} {'Accuracy':>10} {'ROC AUC':>10}")
print("-" * 55)
print(f"{'XGBoost':<20} {'94.1%':>10} {'0.979':>10}")
print(f"{'LSTM (PyTorch)':<20} {'100.0%':>10} {'1.000':>10}")
print("-" * 55)
print("""
Note: LSTM achieved perfect results due to the inherently 
clean churn signal in synthetic data. In production with 
real data, churn is driven by unpredictable human behaviour 
and competitive factors that naturally reduce model accuracy. 
The XGBoost result of 94.1% accuracy and 0.979 ROC AUC is 
a more representative benchmark for real world performance.
""")